# Match ND-LAr (SPINE) and TMS truth, then view vertices or individual particles

Matching follows `match_spine_to_truth_spill.py` (see `TRUTH_SPILL_MATCHING.md`):
1. **Method 1 (IDs)**: SPINE `orig_id` is not the Truth_Spill vertex ID, so interactions are matched by their
   (track ID, PDG) signature with an order-preserving alignment, separately for normal and throughgoing
   (1e15-flagged) interactions. Cross-check only: reliable for normal interactions, not for throughgoing ones.
2. **Method 2 (position)**: Truth_Spill `TrueVtxX/Y/Z` vs SPINE `ixn.vertex` (cm → mm), within `VERTEX_TOL_MM`.
   **This is the reference.**
3. **Method 2.5 (particles/tracks)**: within each Method 2 vertex, Truth_Spill particles are matched to SPINE
   particles by track ID (confirmed by PDG and birth position), and TMS reco tracks are linked to them through
   `Truth_Info.RecoTrackPrimaryParticleIndex`.

Two plotting entry points at the end: **by vertex** (`draw_vertices`) and **by particle/track match**
(`draw_particles`).

Run inside the `spine-nb` container (`/data` = `~/Downloads`).

In [7]:
import sys

SOFTWARE_DIR = '/sdf/data/neutrino/software/spine/src/' # Change this path to your software install

# Set software directory
sys.path.append(SOFTWARE_DIR)

In [8]:
import numpy as np
import pandas as pd
import uproot
import yaml
import plotly.graph_objects as go
from spine.driver import Driver

In [9]:
data_dir = "./"
spine_file = data_dir + "MiniProdN5p3_NDComplex_FHC.spine.full.sanddrift.0000005.MLRECO_SPINE.hdf5"
tms_file = data_dir + "MiniProdN5p3_NDComplex_FHC.tmsreco.full.sanddrift.v4.0000005.TMSRECO.root"

VERTEX_TOL_MM = 1.0
PARTICLE_TOL_MM = 1.0
THROUGHGOING_FLAG = 10**15  # ID flag on external (throughgoing) vertices, in both SPINE orig_id and Truth_Spill IDs

## Colour palette

In [10]:
NDLAr_colour = '#e63946'

strong_colours = ['#E69F00', '#0072B2', '#009E73', '#CC79A7', '#D55E00', '#56B4E9', '#F0E442']
light_colours = ['#f0c975', '#70b1d9', '#7ecdb9', '#e0adc9', '#e8a176', '#a2d9f7', '#f7f2a1']

## Load files

In [11]:
tms_root = uproot.open(tms_file)
tms = tms_root["Truth_Spill"].arrays(
    ["SpillNo", "TrueVtxGlobalID", "TrueVtxX", "TrueVtxY", "TrueVtxZ",
     "VertexGlobalID", "TrackId", "PDG", "IsPrimary", "BirthPosition"], library="np")
spill_to_entry = {int(s): e for e, s in enumerate(tms["SpillNo"])}

# Truth_Info (one entry per TMS slice): true hits tagged by global vertex ID and by the
# Truth_Spill track ID they descend from, plus the truth link of each reco track
tms_info = tms_root["Truth_Info"].arrays(
    ["SpillNo", "TrueHitX", "TrueHitY", "TrueHitZ", "TrueHitVertexId", "TrueHitPrimaryId",
     "RecoTrackN", "RecoTrackPrimaryParticleIndex"], library="np")

# Reco_Tree lines up entry by entry with Truth_Info; TrackHitPos[track][hit] = (x, y, z, t)
tms_reco = tms_root["Reco_Tree"].arrays(["nTracks", "nHits", "TrackHitPos"], library="np")

cfg = f'''
base:
  verbosity: warning
build:
  mode: both
  fragments: false
  particles: true
  interactions: true
  units: cm
io:
  reader:
    file_keys: {spine_file}
    skip_unknown_attrs: True
    name: hdf5
'''
driver = Driver(yaml.safe_load(cfg))
spine_events = [driver.process(i) for i in range(len(driver))]
print(f"{len(tms['SpillNo'])} Truth_Spill entries, {len(spine_events)} SPINE events")

7 Truth_Spill entries, 7 SPINE events


In [12]:
# Index TMS true hits by (Truth_Spill entry, global vertex ID): xyz (mm) and primary track ID per hit
_hits = {}
for s, x, y, z, vid, pid in zip(tms_info["SpillNo"], tms_info["TrueHitX"], tms_info["TrueHitY"],
                                tms_info["TrueHitZ"], tms_info["TrueHitVertexId"], tms_info["TrueHitPrimaryId"]):
    entry = spill_to_entry[int(s)]
    for g in np.unique(vid):
        sel = vid == g #make a mask that only selects vid == g for each unique value in vid
        _hits.setdefault((entry, int(g)), []).append((np.stack([x[sel], y[sel], z[sel]], 1), pid[sel]))
tms_hits_by_vertex = {key: (np.concatenate([h[0] for h in v]), np.concatenate([h[1] for h in v]))
                      for key, v in _hits.items()}

# (Truth_Spill entry, Truth_Spill particle index) -> list of (Truth_Info entry, reco track index)
reco_tracks_by_particle = {}
for i, (s, n_trk) in enumerate(zip(tms_info["SpillNo"], tms_info["RecoTrackN"])):
    for k in range(n_trk):
        p_idx = int(tms_info["RecoTrackPrimaryParticleIndex"][i][k])
        reco_tracks_by_particle.setdefault((spill_to_entry[int(s)], p_idx), []).append((i, k))


def tms_true_hits(entry, gid, track_id=None):
    """TMS true hits (mm) of a vertex, optionally only those descending from one Truth_Spill track ID."""
    xyz, pid = tms_hits_by_vertex.get((entry, gid), (np.empty((0, 3)), np.empty(0, int)))
    return xyz if track_id is None else xyz[pid == track_id]


def reco_track_hits(info_entry, k):
    """Hit positions (mm) of TMS reco track k in Truth_Info / Reco_Tree entry info_entry."""
    n = int(tms_reco["nHits"][info_entry][k])
    return np.asarray(tms_reco["TrackHitPos"][info_entry][k][:n, :3])

## Method 1: match by (track ID, PDG) signature + order-preserving alignment

Normal and throughgoing interactions are separate ID sequences, so each is aligned on its own.

In [13]:
def tms_signatures(entry, throughgoing=False):
    """Per Truth_Spill vertex of one kind (sorted by global ID): (global ID, set of (track, pdg))."""
    gids = np.sort(tms["TrueVtxGlobalID"][entry])
    gids = gids[(gids >= THROUGHGOING_FLAG) == throughgoing]
    sig = {int(g): set() for g in gids}
    for g, trk, pdg in zip(tms["VertexGlobalID"][entry], tms["TrackId"][entry], tms["PDG"][entry]):
        if int(g) in sig:
            sig[int(g)].add((int(trk), int(pdg)))
    return [(int(g), sig[int(g)]) for g in gids]


def spine_signatures(data, throughgoing=False):
    """SPINE interactions of one kind, in orig_id order: (ixn, set of (track, pdg))."""
    ixns = sorted((x for x in data["truth_interactions"]
                   if (int(x.orig_id) >= THROUGHGOING_FLAG) == throughgoing),
                  key=lambda x: int(x.orig_id))
    sig = {x.id: set() for x in ixns}
    for p in data["truth_particles"]:
        if p.interaction_id in sig:
            sig[p.interaction_id].add((int(p.track_id), int(p.pdg_code)))
    return [(x, sig[x.id]) for x in ixns]


def align(spine_sig, tms_sig):
    """Order-preserving assignment; returns (total score, {spine position: tms position})."""
    n, m = len(spine_sig), len(tms_sig)
    score = np.zeros((n, m))
    for i, (_, s) in enumerate(spine_sig):
        for j, (_, t) in enumerate(tms_sig):
            score[i, j] = (len(t & s) - 0.5 * len(t - s)) / max(len(t), 1)
    best = np.full((n + 1, m + 1), -1e9)
    best[0, :] = 0
    took = np.zeros((n + 1, m + 1), bool)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            take = best[i - 1, j - 1] + score[i - 1, j - 1]
            best[i, j] = max(best[i, j - 1], take)
            took[i, j] = take >= best[i, j - 1]
    i, j, assign = n, m, {}
    while i > 0 and j > 0:
        if took[i, j]:
            assign[i - 1] = j - 1
            i, j = i - 1, j - 1
        else:
            j -= 1
    return best[n, m], assign


def match_by_ids():
    """{(SPINE event, ixn id): (Truth_Spill entry, global vertex ID)}"""
    n_entries = len(tms["SpillNo"])
    tms_sigs = {tg: [tms_signatures(e, tg) for e in range(n_entries)] for tg in (False, True)}
    out = {}
    for ev, data in enumerate(spine_events):
        s_sig = {tg: spine_signatures(data, tg) for tg in (False, True)}
        results = {tg: [align(s_sig[tg], t_sig) for t_sig in tms_sigs[tg]] for tg in (False, True)}
        entry = int(np.argmax([results[False][e][0] + results[True][e][0] for e in range(n_entries)]))
        for tg in (False, True):
            for i, j in results[tg][entry][1].items():
                out[(ev, int(s_sig[tg][i][0].id))] = (entry, tms_sigs[tg][entry][j][0])
    return out


id_matches = match_by_ids()
print(f"{len(id_matches)} SPINE interactions assigned a vertex by IDs")

700 SPINE interactions assigned a vertex by IDs


## Method 2: match by vertex position (reference)

In [14]:
def tms_vertex_xyz(entry):
    return np.stack([tms["TrueVtxX"][entry], tms["TrueVtxY"][entry], tms["TrueVtxZ"][entry]], axis=1)


def match_by_position(tol_mm=VERTEX_TOL_MM):
    """{(SPINE event, ixn id): (Truth_Spill entry, global vertex ID, distance mm)}"""
    tms_xyz = [tms_vertex_xyz(e) for e in range(len(tms["SpillNo"]))]
    out = {}
    for ev, data in enumerate(spine_events):
        for ixn in data["truth_interactions"]:
            pos = np.asarray(ixn.vertex) * 10.0  # cm -> mm
            best = None
            for entry, xyz in enumerate(tms_xyz):
                dist = np.linalg.norm(xyz - pos, axis=1)
                j = int(dist.argmin())
                if dist[j] <= tol_mm and (best is None or dist[j] < best[2]):
                    best = (entry, int(tms["TrueVtxGlobalID"][entry][j]), float(dist[j]))
            if best is not None:
                out[(ev, int(ixn.id))] = best
    return out


position_matches = match_by_position()
print(f"{len(position_matches)} SPINE interactions matched to a Truth_Spill vertex within {VERTEX_TOL_MM} mm")

700 SPINE interactions matched to a Truth_Spill vertex within 1.0 mm


## Cross-check: Method 1 vs Method 2

In [15]:
agree = {False: 0, True: 0}
total = {False: 0, True: 0}
for key, (entry, gid) in id_matches.items():
    if key not in position_matches:
        continue
    tg = gid >= THROUGHGOING_FLAG
    total[tg] += 1
    agree[tg] += position_matches[key][:2] == (entry, gid)
print(f"ID matches that agree with position matches: {agree[False] + agree[True]}/{total[False] + total[True]} "
      f"(normal {agree[False]}/{total[False]}, throughgoing {agree[True]}/{total[True]})")

ID matches that agree with position matches: 274/700 (normal 195/196, throughgoing 79/504)


## Method 2.5: particle and track matching within each Method 2 vertex

Status per Truth_Spill particle: `matched` (same track ID, PDG and birth position within `PARTICLE_TOL_MM`),
`rejected` (same track ID, PDG or position disagree), `tms_only` (no SPINE particle with that track ID; mostly
neutrals, or particles that never enter ND-LAr).

In [16]:
def match_particles(entry, gid, ev, ixn_id, tol_mm=PARTICLE_TOL_MM):
    spine_parts = {int(p.track_id): p for p in spine_events[ev]["truth_particles"] if p.interaction_id == ixn_id}
    out = []
    for idx in np.where(tms["VertexGlobalID"][entry] == gid)[0]:
        trk, pdg = int(tms["TrackId"][entry][idx]), int(tms["PDG"][entry][idx])
        birth = np.asarray(tms["BirthPosition"][entry][idx][:3], dtype=float)
        rec = {"tms_particle_idx": int(idx), "track_id": trk, "tms_pdg": pdg,
               "is_primary": bool(tms["IsPrimary"][entry][idx]),
               "reco_tracks": reco_tracks_by_particle.get((entry, int(idx)), []),
               "spine_particle_id": None, "spine_pdg": None, "birth_dist_mm": None}
        p = spine_parts.get(trk)
        if p is None:
            rec["status"] = "tms_only"
        else:
            dist = float(np.linalg.norm(np.asarray(p.position) * 10.0 - birth))  # cm -> mm
            rec.update(spine_particle_id=int(p.id), spine_pdg=int(p.pdg_code), birth_dist_mm=dist)
            rec["status"] = "matched" if (int(p.pdg_code) == pdg and dist <= tol_mm) else "rejected"
        out.append(rec)
    return out


particle_matches = {key: match_particles(entry, gid, *key) for key, (entry, gid, _) in position_matches.items()}

records = [r for recs in particle_matches.values() for r in recs]
print(f"Truth_Spill particles in matched vertices: {len(records)}",
      {s: sum(r["status"] == s for r in records) for s in ("matched", "rejected", "tms_only")})
n_reco = sum(len(r["reco_tracks"]) for r in records)
print(f"TMS reco tracks from matched vertices: {n_reco}, linked to a SPINE particle: "
      f"{sum(len(r['reco_tracks']) for r in records if r['status'] == 'matched')}")

Truth_Spill particles in matched vertices: 5276 {'matched': 1065, 'rejected': 0, 'tms_only': 4211}
TMS reco tracks from matched vertices: 255, linked to a SPINE particle: 228


## Vertex table

In [17]:
rows = []
for (ev, ixn_id), (entry, gid, dist) in sorted(position_matches.items()):
    ixn = spine_events[ev]["truth_interactions"][ixn_id]
    id_match = id_matches.get((ev, ixn_id))
    recs = particle_matches[(ev, ixn_id)]
    rows.append({
        "SPINE_event": ev,
        "ixn_id": ixn_id,
        "orig_id": int(ixn.orig_id),
        "throughgoing": gid >= THROUGHGOING_FLAG,
        "TS_entry": entry,
        "TS_vtx_global_id": gid,
        "dist_mm": round(dist, 4),
        "id_method_agrees": None if id_match is None else id_match == (entry, gid),
        "n_particles_matched": sum(r["status"] == "matched" for r in recs),
        "n_reco_tracks_linked": sum(len(r["reco_tracks"]) for r in recs if r["status"] == "matched"),
        "n_NDLAr_points": len(ixn.points),
        "n_TMS_hits": len(tms_true_hits(entry, gid)),
    })
match_df = pd.DataFrame(rows)
print("ID method agreement (normal):", match_df[~match_df.throughgoing].id_method_agrees.value_counts().to_dict())
print("ID method agreement (throughgoing):", match_df[match_df.throughgoing].id_method_agrees.value_counts().to_dict())
pd.set_option("display.max_rows", 50)
match_df

ID method agreement (normal): {True: 195, False: 1}
ID method agreement (throughgoing): {False: 425, True: 79}


,SPINE_event,ixn_id,orig_id,throughgoing,TS_entry,TS_vtx_global_id,dist_mm,id_method_agrees,n_particles_matched,n_reco_tracks_linked,n_NDLAr_points,n_TMS_hits
0,0,0,50000000,False,0,50000006,0.0005,True,3,0,1928,4
1,0,1,50000001,False,0,50000023,0.0000,True,3,0,498,7
2,0,2,50000002,False,0,50000032,0.0001,True,4,0,1522,0
3,0,3,50000003,False,0,50000047,0.0005,True,1,0,1866,0
4,0,4,50000004,False,0,50000053,0.0005,True,4,1,1443,41
...,...,...,...,...,...,...,...,...,...,...,...,...
695,6,94,1000000059001534,True,6,1000000059003097,0.0001,False,0,0,136,0
696,6,95,1000000059001547,True,6,1000000059003101,0.0005,False,1,1,3174,76
697,6,96,1000000059001642,True,6,1000000059003114,0.0003,False,1,0,2474,0
698,6,97,1000000059001677,True,6,1000000059003121,0.0001,False,1,0,2398,2


In [18]:
# Vertices with hits in both detectors are the interesting ones for track matching
both = match_df[(match_df.n_NDLAr_points > 0) & (match_df.n_TMS_hits > 0)]
print(f"{len(both)} matched vertices have hits in both ND-LAr and TMS")
both

391 matched vertices have hits in both ND-LAr and TMS


,SPINE_event,ixn_id,orig_id,throughgoing,TS_entry,TS_vtx_global_id,dist_mm,id_method_agrees,n_particles_matched,n_reco_tracks_linked,n_NDLAr_points,n_TMS_hits
0,0,0,50000000,False,0,50000006,0.0005,True,3,0,1928,4
1,0,1,50000001,False,0,50000023,0.0000,True,3,0,498,7
4,0,4,50000004,False,0,50000053,0.0005,True,4,1,1443,41
5,0,5,50000005,False,0,50000069,0.0001,True,4,1,2054,84
9,0,9,50000009,False,0,50000123,0.0001,True,9,1,707,499
...,...,...,...,...,...,...,...,...,...,...,...,...
688,6,87,1000000059001231,True,6,1000000059003056,0.0010,True,1,1,2627,43
690,6,89,1000000059001279,True,6,1000000059003061,0.0002,False,1,1,2934,121
694,6,93,1000000059001483,True,6,1000000059003094,0.0001,False,1,1,2386,106
696,6,95,1000000059001547,True,6,1000000059003101,0.0005,False,1,1,3174,76


## Particle table

One row per **matched** particle (Method 2.5). `n_TMS_hits` counts TMS true hits whose `TrueHitPrimaryId` is this
track ID, so it includes hits from the particle's descendants; `n_NDLAr_points` is the SPINE particle's own points.

In [19]:
prow = []
for (ev, ixn_id), recs in sorted(particle_matches.items()):
    entry, gid, _ = position_matches[(ev, ixn_id)]
    for r in recs:
        if r["status"] != "matched":
            continue
        p = spine_events[ev]["truth_particles"][r["spine_particle_id"]]
        prow.append({
            "SPINE_event": ev,
            "ixn_id": ixn_id,
            "TS_entry": entry,
            "TS_vtx_global_id": gid,
            "throughgoing": gid >= THROUGHGOING_FLAG,
            "track_id": r["track_id"],
            "pdg": r["tms_pdg"],
            "is_primary": r["is_primary"],
            "spine_particle_id": r["spine_particle_id"],
            "birth_dist_mm": round(r["birth_dist_mm"], 4),
            "n_NDLAr_points": len(p.points),
            "n_TMS_hits": len(tms_true_hits(entry, gid, r["track_id"])),
            "reco_tracks": r["reco_tracks"],
            "n_reco_tracks": len(r["reco_tracks"]),
        })
particle_df = pd.DataFrame(prow)
print(f"{len(particle_df)} matched particles")
particle_df.head(20)

1065 matched particles


,SPINE_event,ixn_id,TS_entry,TS_vtx_global_id,throughgoing,track_id,pdg,is_primary,spine_particle_id,birth_dist_mm,n_NDLAr_points,n_TMS_hits,reco_tracks,n_reco_tracks
0,0,0,0,50000006,False,0,13,True,8,0.0005,948,0,[],0
1,0,0,0,50000006,False,2,2212,True,9,0.0005,94,0,[],0
2,0,0,0,50000006,False,3,211,True,10,0.0005,167,0,[],0
3,0,1,0,50000023,False,1,13,True,26,0.0000,409,7,[],0
4,0,1,0,50000023,False,3,2212,True,27,0.0000,28,0,[],0
5,0,1,0,50000023,False,4,211,True,28,0.0000,28,0,[],0
6,0,2,0,50000032,False,0,13,True,31,0.0001,1038,0,[],0
7,0,2,0,50000032,False,2,2212,True,32,0.0001,74,0,[],0
8,0,2,0,50000032,False,3,2212,True,33,0.0001,101,0,[],0
9,0,2,0,50000032,False,4,-211,True,34,0.0001,23,0,[],0


In [20]:
# Particles seen in both detectors: ND-LAr points and TMS hits, and the ones with a TMS reco track
particles_both = particle_df[(particle_df.n_NDLAr_points > 0) & (particle_df.n_TMS_hits > 0)]
particles_reco = particle_df[particle_df.n_reco_tracks > 0]
print(f"{len(particles_both)} matched particles have hits in both detectors; "
      f"{len(particles_reco)} have a TMS reco track")
particles_reco

373 matched particles have hits in both detectors; 228 have a TMS reco track


,SPINE_event,ixn_id,TS_entry,TS_vtx_global_id,throughgoing,track_id,pdg,is_primary,spine_particle_id,birth_dist_mm,n_NDLAr_points,n_TMS_hits,reco_tracks,n_reco_tracks
11,0,4,0,50000053,False,0,13,True,58,0.0005,853,41,"[(9, 0)]",1
15,0,5,0,50000069,False,0,13,True,71,0.0001,1435,84,"[(12, 0)]",1
34,0,9,0,50000123,False,3,211,True,140,0.0001,59,276,"[(27, 0)]",1
42,0,10,0,50000126,False,0,13,True,151,0.0000,526,35,"[(28, 0)]",1
50,0,11,0,50000144,False,0,13,True,171,0.0001,424,78,"[(35, 0)]",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1053,6,86,6,1000000059003051,True,1,13,True,764,0.0002,2602,92,"[(848, 0)]",1
1054,6,87,6,1000000059003056,True,0,-13,True,772,0.0010,2512,39,"[(853, 0)]",1
1055,6,89,6,1000000059003061,True,0,13,True,778,0.0002,2650,121,"[(856, 0)]",1
1060,6,93,6,1000000059003094,True,0,13,True,835,0.0001,2103,106,"[(864, 0)]",1


## Figure setup and detector boxes

In [21]:
import plotly.express as px
from IPython.display import display

# Extra colours once more than 7 particles are plotted (gray is reserved for unmatched particles)
extra_strong = px.colors.qualitative.Dark24
extra_light = px.colors.qualitative.Light24
UNMATCHED_SPINE_GRAY = '#7f7f7f'
UNMATCHED_TMS_GRAY = '#c7c7c7'


def colour_pair(idx):
    """(strong, light) colours for the idx-th plotted object."""
    if idx < len(strong_colours):
        return strong_colours[idx], light_colours[idx]
    k = (idx - len(strong_colours)) % len(extra_strong)
    return extra_strong[k], extra_light[k]


def record(legend, colour, marker, item, shown_as, count=None):
    """Add one row to the legend table (legend is a list, or None to skip)."""
    if legend is not None:
        legend.append({"colour": colour, "marker": marker, "item": item, "shown as": shown_as, "count": count})


def legend_table_styler(legend):
    """The legend rows as a styled table with a colour swatch per row."""
    df = pd.DataFrame(legend)
    df.insert(0, "", "")
    return (df.style
            .apply(lambda col: [f"background-color: {c}" for c in df["colour"]], subset=[""])
            .hide(axis="index")
            .format({"count": lambda v: "" if v is None or pd.isna(v) else f"{int(v)}"}))


def show_legend_table(legend):
    """Display the legend rows as a table with a colour swatch per row."""
    display(legend_table_styler(legend))


def save_html_page(figure, path, legend=None):
    """Write the figure (and the legend table, if given) to one self-contained HTML file.

    plotly.js is embedded, so the file opens offline in any browser, with rotation, hover and the
    layer on/off buttons working.
    """
    import html as _html
    title = _html.escape(figure.layout.title.text or "plot")
    body = figure.to_html(full_html=False, include_plotlyjs=True)
    if legend:
        body += "<h3>Legend</h3>" + legend_table_styler(legend).to_html()
    page = (f"<!DOCTYPE html><html><head><meta charset='utf-8'><title>{title}</title>"
            "<style>body{font-family:sans-serif;margin:16px}"
            "table{border-collapse:collapse;font-size:13px}"
            "td,th{border:1px solid #ccc;padding:2px 6px;text-align:left}</style></head>"
            f"<body>{body}</body></html>")
    with open(path, "w") as f:
        f.write(page)
    print(f"Saved {path} ({len(page.encode()) / 1e6:.1f} MB)")


def plot_box(fig, x_min, x_max, y_min, y_max, z_min, z_max, color, name, legend=None):
    # One continuous path over every edge; None breaks the line between disconnected edges
    x = [x_min, x_max, x_max, x_min, x_min, x_min, x_max, x_max, x_min, x_min, None, x_max, x_max, None, x_min, x_min, None, x_max, x_max]
    y = [y_min, y_min, y_max, y_max, y_min, y_min, y_min, y_max, y_max, y_min, None, y_min, y_min, None, y_max, y_max, None, y_max, y_max]
    z = [z_min, z_min, z_min, z_min, z_min, z_max, z_max, z_max, z_max, z_max, None, z_min, z_max, None, z_min, z_max, None, z_min, z_max]
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="lines",
                               line=dict(color=color, width=3), name=name, showlegend=True))
    record(legend, color, "box outline", name, "detector volume")


def setup_figure(figure, title, legend=None):
    figure.update_layout(
        width=1600, height=900,
        scene=dict(
            xaxis=dict(title="x [mm]", range=[-4500, 4500]),
            yaxis=dict(title="y [mm]", range=[-4000, 1000]),
            zaxis=dict(title="z [mm]", range=[0, 20000]),
            camera=dict(eye=dict(x=2.0, y=4.0, z=0.0), up=dict(x=0, y=-1, z=0)),  # Y is up
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=4),
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        showlegend=True,
        legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top',
                    bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='Black', borderwidth=1,
                    itemsizing='constant'),
        title=dict(text=title, x=0.5, xanchor='center'),
    )
    figure.data = []
    # Geometry from TMS_Constants.h (mm)
    plot_box(figure, -3730, 3730, -3702.23, 997.77, 11177.5, 18502.5, "blue", "TMS detector", legend)
    plot_box(figure, -3478.48, 3478.48, -2166.71, 829.282, 4179.24, 9135.88, "black", "ND-LAr detector", legend)


def add_points(figure, xyz, colour, item, shown_as, size=2, legend=None):
    if len(xyz):
        figure.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                                      marker=dict(size=size, color=colour, symbol='circle'),
                                      name=f"{item} {shown_as}", hoverlabel_namelength=-1))
        record(legend, colour, "points", item, shown_as, len(xyz))


def add_vertex_markers(figure, ev, ixn_id, entry, gid, label, legend=None):
    v = np.asarray(spine_events[ev]["truth_interactions"][ixn_id].vertex) * 10.0
    figure.add_trace(go.Scatter3d(
        x=[v[0]], y=[v[1]], z=[v[2]], mode="markers",
        marker=dict(size=5, color=NDLAr_colour, symbol='square-open'),
        name=f"{label} SPINE vertex [{v[0]:.1f}, {v[1]:.1f}, {v[2]:.1f}]", hoverlabel_namelength=-1))
    record(legend, NDLAr_colour, "open square", label, f"SPINE vertex [{v[0]:.1f}, {v[1]:.1f}, {v[2]:.1f}] mm")
    k = int(np.where(tms["TrueVtxGlobalID"][entry] == gid)[0][0])
    tx, ty, tz = tms["TrueVtxX"][entry][k], tms["TrueVtxY"][entry][k], tms["TrueVtxZ"][entry][k]
    figure.add_trace(go.Scatter3d(
        x=[tx], y=[ty], z=[tz], mode="markers",
        marker=dict(size=5, color='black', symbol='cross'),
        name=f"{label} Truth_Spill vertex [{tx:.1f}, {ty:.1f}, {tz:.1f}]", hoverlabel_namelength=-1))
    record(legend, 'black', "cross", label, f"Truth_Spill vertex [{tx:.1f}, {ty:.1f}, {tz:.1f}] mm")


## Plot by vertex

For each chosen `match_df` row: all ND-LAr points of the SPINE interaction, all TMS true hits of the vertex, and
both vertex markers.

In [22]:
def plot_matched_vertex(figure, row, colour_idx):
    strong, light = colour_pair(colour_idx)
    ev, ixn_id, entry, gid = int(row.SPINE_event), int(row.ixn_id), int(row.TS_entry), int(row.TS_vtx_global_id)
    label = f"(ev {ev} ixn {ixn_id} / vtx {gid})"
    ixn = spine_events[ev]["truth_interactions"][ixn_id]
    add_points(figure, np.asarray(ixn.points) * 10.0, strong, label, "ND-LAr hits")  # cm -> mm
    hits = tms_true_hits(entry, gid)
    add_points(figure, hits, light, label, f"TMS true hits ({len(hits)})")
    add_vertex_markers(figure, ev, ixn_id, entry, gid, label)


def draw_vertices(rows, save_html=None, return_fig=False):
    """Plot the matched vertices in the given match_df rows.

    save_html:  path of an HTML file to also write the plot to (self-contained, opens offline)
    return_fig: return the plotly figure (assign it, e.g. fig = draw_vertices(...), to avoid
                Jupyter displaying it twice)
    """
    fig = go.Figure()
    setup_figure(fig, f"Matched vertices (match_df rows): {[int(r) for r in rows]}")
    for c, r in enumerate(rows):
        plot_matched_vertex(fig, match_df.loc[r], c)
    fig.show()
    if save_html:
        save_html_page(fig, save_html)
    if return_fig:
        return fig


In [52]:
# Choose match_df rows to plot (default: first few vertices with hits in both detectors)
vertex_rows_to_plot = list(both.index[:3]) if len(both) else [0]
draw_vertices(vertex_rows_to_plot)

## Plot by particle / track match

For each chosen `particle_df` row: the SPINE particle's ND-LAr points, the TMS true hits descending from the same
track ID, every TMS reco track linked to it (hits joined by a line), and (optionally) the vertex markers.

`draw_particles(rows, show_vertex=True, show_legend=True, show_other_particles=False, show_table=True, given_title=None, show_all_activity=False, toggles=True, save_html=None, return_fig=False, tracks_only=False, throughgoing_only=False)`:
- **`show_table`** prints a table below the plot listing everything in the legend, with a colour swatch per row.
- **`show_legend=False`** removes the legend from the plot entirely, so the table is the reference.
- **`show_other_particles=True`** also plots the other particles at the same vertices: other *matched* particles in
  colour (same style as the chosen ones), and everything *unmatched* in gray, grouped per vertex:
  SPINE particles with no Truth_Spill counterpart (dark gray, ND-LAr) and TMS true hits not belonging to a
  matched particle (light gray, TMS).
- **`show_all_activity=True`** adds a further layer (and turns on `show_other_particles`): everything else in the same
  spill(s), in faint gray — every other SPINE interaction's ND-LAr points, every other Truth_Spill vertex's TMS true
  hits, and every TMS reco track not already drawn. Shown as one trace per kind per spill.
- **`given_title`** sets the plot title.
- **`tracks_only=True`** starts with only track-shaped particles shown, in **every** layer that is on (chosen,
  other particles, all activity). Track = SPINE shape `track` for ND-LAr particles; TMS reco tracks always count
  as tracks; TMS true hits of TMS-only particles (no SPINE shape) count as track-like if their PDG is μ±, π±, K± or
  p/p̄. It does **not** turn any layer on: with "All other activity" off, it only filters what is already shown.
- **`throughgoing_only=True`** starts with only particles from throughgoing (1e15-flagged) vertices shown, in
  every layer, plus their vertex markers. Also does not turn any layer on.
- **`save_html="file.html"`** also writes the plot, with the legend table below it, to one self-contained HTML file
  (plotly.js embedded, so it opens offline in any browser and the layer buttons keep working). Hidden layers are
  saved too, so with toggles on the file can be tens of MB. `draw_vertices(rows, save_html=...)` saves its plot the
  same way (no table).
- **`return_fig=True`** returns the plotly figure for further changes; assign it (`fig = draw_particles(...)`) so
  Jupyter doesn't display it twice.
- **`toggles=True`** (default) builds all three optional layers (vertex markers, other particles at these vertices,
  all other activity) and adds an on/off button for each above the plot (top row), plus **Tracks only** and
  **Throughgoing only** filter buttons (second row). A trace shows only if its layer is on and no active filter
  hides it, so e.g. Tracks only + All other activity shows only the tracks in the activity layer. Each button works independently: the first
  click flips its layer, the second click flips it back. `show_vertex`, `show_other_particles` and `show_all_activity`
  then only set what is visible at first. The table lists every layer, with columns saying which layer each row
  belongs to, whether it counts as a track and as throughgoing, and whether it was visible at the start. `toggles=False` builds only what is asked for (faster, no buttons).

Colours follow the 7-colour palette for the first 7 particles, then plotly's Dark24/Light24.

Implementation note: the layer buttons switch traces' `visible` property, while the filters hide traces by making
them transparent (Tracks only: trace opacity; Throughgoing only: marker opacity and line width), so all five
buttons combine. Filtered traces stay in the plot legend and may still show hover labels; the table's
"visible at start" column accounts for the filters.


In [ ]:
TRACK_SHAPE = 1  # SPINE semantic shape label for tracks
# TMS true hits whose particle has no SPINE shape (TMS-only particles) are classed as track-like by PDG:
TRACK_LIKE_PDGS = {13, -13, 211, -211, 321, -321, 2212, -2212}


def tag_traces(figure, tags, track, throughgoing):
    """Tag every trace added since the last tag. track / throughgoing: True, False, or None (not filtered)."""
    if tags is not None:
        while len(tags) < len(figure.data):
            tags.append({"track": track, "throughgoing": throughgoing})


def tms_hit_is_track_like(entry, gid, pid):
    """Per TMS true hit (primary track IDs pid at vertex gid): PDG of that Truth_Spill particle is track-like."""
    m = tms["VertexGlobalID"][entry] == gid
    pdg_of = dict(zip(tms["TrackId"][entry][m].tolist(), tms["PDG"][entry][m].tolist()))
    return np.array([pdg_of.get(int(t), 0) in TRACK_LIKE_PDGS for t in pid], dtype=bool)


def plot_matched_particle(figure, row, colour_idx, show_vertex=True, legend=None, tags=None):
    strong, light = colour_pair(colour_idx)
    ev, ixn_id, entry, gid = int(row.SPINE_event), int(row.ixn_id), int(row.TS_entry), int(row.TS_vtx_global_id)
    label = f"(spill {ev} ixn {ixn_id} track {row.track_id}, PDG {row.pdg})"
    p = spine_events[ev]["truth_particles"][int(row.spine_particle_id)]
    is_track, is_tg = int(p.shape) == TRACK_SHAPE, bool(row.throughgoing)
    add_points(figure, np.asarray(p.points) * 10.0, strong, label, "ND-LAr points", legend=legend)  # cm -> mm
    hits = tms_true_hits(entry, gid, int(row.track_id))
    add_points(figure, hits, light, label, f"TMS true hits ({len(hits)})", legend=legend)
    for info_entry, k in row.reco_tracks:
        rh = reco_track_hits(info_entry, k)
        if len(rh):
            shown_as = f"TMS reco track (slice {info_entry}, #{k}, {len(rh)} hits)"
            figure.add_trace(go.Scatter3d(
                x=rh[:, 0], y=rh[:, 1], z=rh[:, 2], mode="lines+markers",
                line=dict(color=strong, width=4), marker=dict(size=3, color=strong, symbol='diamond'),
                name=f"{label} {shown_as}", hoverlabel_namelength=-1))
            record(legend, strong, "line + diamonds", label, shown_as, len(rh))
    tag_traces(figure, tags, is_track, is_tg)
    if show_vertex:
        add_vertex_markers(figure, ev, ixn_id, entry, gid, f"(ev {ev} ixn {ixn_id})", legend)
        tag_traces(figure, tags, None, is_tg)


def plot_unmatched_at_vertex(figure, ev, ixn_id, entry, gid, legend=None, tags=None):
    """Gray: SPINE particles with no Truth_Spill match, and TMS true hits not from a matched particle.
    Split into track / non-track traces so the tracks-only filter can act on them."""
    matched = [r for r in particle_matches[(ev, ixn_id)] if r["status"] == "matched"]
    matched_ids = {r["spine_particle_id"] for r in matched}
    matched_tracks = {r["track_id"] for r in matched}
    is_tg = gid >= THROUGHGOING_FLAG
    label = f"(spill {ev} ixn {ixn_id}) unmatched"
    spine_only = [p for p in spine_events[ev]["truth_particles"]
                  if p.interaction_id == ixn_id and int(p.id) not in matched_ids and len(p.points)]
    for is_track in (True, False):
        group = [p for p in spine_only if (int(p.shape) == TRACK_SHAPE) == is_track]
        if group:
            pts = np.concatenate([np.asarray(p.points) for p in group]) * 10.0  # cm -> mm
            add_points(figure, pts, UNMATCHED_SPINE_GRAY, label,
                       f"SPINE {'track' if is_track else 'non-track'} particles, ND-LAr points "
                       f"({len(group)} particles)", legend=legend)
            tag_traces(figure, tags, is_track, is_tg)
    xyz, pid = tms_hits_by_vertex.get((entry, gid), (np.empty((0, 3)), np.empty(0, int)))
    other = ~np.isin(pid, list(matched_tracks))
    track_like = tms_hit_is_track_like(entry, gid, pid)
    for is_track in (True, False):
        sel = other & (track_like == is_track)
        add_points(figure, xyz[sel], UNMATCHED_TMS_GRAY, label,
                   f"TMS true hits, {'track-like' if is_track else 'non-track-like'} PDG "
                   f"({int(sel.sum())}, from {len(np.unique(pid[sel]))} track IDs)", legend=legend)
        tag_traces(figure, tags, is_track, is_tg)


# Faint grays for activity outside the chosen vertices (lighter than the unmatched-at-vertex grays)
OTHER_NDLAR_GRAY = '#b5b5b5'
OTHER_TMS_GRAY = '#dedede'
OTHER_RECO_GRAY = '#9e9e9e'


def _bucket_name(is_track, is_tg):
    return f"{'tracks' if is_track else 'non-tracks'}, {'throughgoing' if is_tg else 'non-throughgoing'}"


def plot_other_activity(figure, spills, skip_vertices, skip_reco_tracks, legend=None, tags=None):
    """Faint gray: all activity in the given spills outside skip_vertices, one trace per
    (track / non-track) x (throughgoing / not) per kind per spill.

    spills:           SPINE event / Truth_Spill entry indices (they are the same index)
    skip_vertices:    {(spill, ixn id)} already drawn; their Truth_Spill vertices are skipped in TMS too
    skip_reco_tracks: {(Truth_Info entry, reco track index)} already drawn
    """
    buckets = [(t, g) for t in (True, False) for g in (True, False)]
    for ev in spills:
        entry = ev
        label = f"(spill {ev}) other activity"
        # ND-LAr: the particles of every SPINE true interaction not already drawn
        tg_of_ixn = {int(x.id): int(x.orig_id) >= THROUGHGOING_FLAG for x in spine_events[ev]["truth_interactions"]
                     if (ev, int(x.id)) not in skip_vertices}
        pts = {b: [] for b in buckets}
        n_ixn = {b: set() for b in buckets}
        for p in spine_events[ev]["truth_particles"]:
            if int(p.interaction_id) in tg_of_ixn and len(p.points):
                b = (int(p.shape) == TRACK_SHAPE, tg_of_ixn[int(p.interaction_id)])
                pts[b].append(np.asarray(p.points))
                n_ixn[b].add(int(p.interaction_id))
        for b in buckets:
            if pts[b]:
                xyz = np.concatenate(pts[b]) * 10.0  # cm -> mm
                figure.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                                              marker=dict(size=1, color=OTHER_NDLAR_GRAY), opacity=0.5,
                                              name=f"{label} ND-LAr points ({_bucket_name(*b)})",
                                              hoverlabel_namelength=-1))
                record(legend, OTHER_NDLAR_GRAY, "points (faint)", label,
                       f"ND-LAr points, {_bucket_name(*b)} ({len(n_ixn[b])} SPINE interactions)", len(xyz))
                tag_traces(figure, tags, *b)
        # TMS: true hits from every Truth_Spill vertex not already drawn
        skip_gids = {position_matches[v][1] for v in skip_vertices if v[0] == ev}
        hits = {b: [] for b in buckets}
        n_vtx = {b: set() for b in buckets}
        for (e, g), (xyz, pid) in tms_hits_by_vertex.items():
            if e != entry or g in skip_gids:
                continue
            track_like = tms_hit_is_track_like(entry, g, pid)
            for is_track in (True, False):
                sel = track_like == is_track
                if sel.any():
                    b = (is_track, g >= THROUGHGOING_FLAG)
                    hits[b].append(xyz[sel])
                    n_vtx[b].add(g)
        for b in buckets:
            if hits[b]:
                xyz = np.concatenate(hits[b])
                figure.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                                              marker=dict(size=1, color=OTHER_TMS_GRAY), opacity=0.5,
                                              name=f"{label} TMS true hits ({_bucket_name(*b)}, by PDG)",
                                              hoverlabel_namelength=-1))
                record(legend, OTHER_TMS_GRAY, "points (faint)", label,
                       f"TMS true hits, {_bucket_name(*b)} by PDG ({len(n_vtx[b])} vertices)", len(xyz))
                tag_traces(figure, tags, *b)
        # TMS reco tracks in this spill not already drawn (None breaks between tracks); all are tracks,
        # throughgoing if their true particle's vertex is
        spill_no = tms["SpillNo"][entry]
        lines = {True: ([], [], [], 0), False: ([], [], [], 0)}
        for i in np.where(tms_info["SpillNo"] == spill_no)[0]:
            for k in range(int(tms_reco["nTracks"][i])):
                if (int(i), k) in skip_reco_tracks:
                    continue
                rh = reco_track_hits(int(i), k)
                if not len(rh):
                    continue
                p_idx = int(tms_info["RecoTrackPrimaryParticleIndex"][i][k]) if k < tms_info["RecoTrackN"][i] else -1
                is_tg = p_idx >= 0 and int(tms["VertexGlobalID"][entry][p_idx]) >= THROUGHGOING_FLAG
                xs, ys, zs, n = lines[is_tg]
                xs += list(rh[:, 0]) + [None]; ys += list(rh[:, 1]) + [None]; zs += list(rh[:, 2]) + [None]
                lines[is_tg] = (xs, ys, zs, n + 1)
        for is_tg in (True, False):
            xs, ys, zs, n_trk = lines[is_tg]
            if n_trk:
                shown_as = f"TMS reco tracks, {'throughgoing' if is_tg else 'non-throughgoing'} ({n_trk})"
                figure.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                                              line=dict(color=OTHER_RECO_GRAY, width=2), opacity=0.6,
                                              name=f"{label} {shown_as}", hoverlabel_namelength=-1))
                record(legend, OTHER_RECO_GRAY, "lines (faint)", label, shown_as, n_trk)
                tag_traces(figure, tags, True, is_tg)


# Toggleable layers: (key, button label)
TOGGLE_LAYERS = [("vertex", "Vertex markers"),
                 ("other", "Other particles at these vertices"),
                 ("activity", "All other activity in spill")]
# Filters that act on every layer: (key, button label)
TOGGLE_FILTERS = [("track", "Tracks only"),
                  ("throughgoing", "Throughgoing only")]


def draw_particles(rows, show_vertex=True, show_legend=True, show_other_particles=False, show_table=True, given_title=None,
                   show_all_activity=False, toggles=True, save_html=None, return_fig=False,
                   tracks_only=False, throughgoing_only=False):
    """Plot the matched particles in the given particle_df rows.

    show_vertex:          start with the SPINE and Truth_Spill vertex markers visible
    show_legend:          draw the plot legend (False: use the table instead)
    show_other_particles: start with the other particles at the same vertices visible; matched ones in colour,
                          unmatched ones in gray
    show_table:           display a table of everything in the legend, with colour swatches
    given_title:          plot title (default: lists the particle_df rows)
    show_all_activity:    start with all other activity in the same spills visible, in faint gray: every other
                          SPINE interaction (ND-LAr), every other Truth_Spill vertex's TMS true hits, and every
                          TMS reco track not already drawn. Implies show_other_particles.
    tracks_only:          start with only track-shaped particles shown, in every layer (SPINE shape == track;
                          TMS true hits of TMS-only particles by PDG; TMS reco tracks count as tracks).
                          Does not turn any layer on.
    throughgoing_only:    start with only particles from throughgoing (1e15-flagged) vertices shown, in every
                          layer. Does not turn any layer on.
    toggles:              build all optional layers and add on/off buttons above the plot for the three layers
                          and the two filters; the show_* / *_only options then only set the starting state.
                          False: only build what is asked for (faster, no buttons).
    save_html:            path of an HTML file to also write the plot to, with the legend table below it
                          (self-contained, opens offline; the buttons keep working)
    return_fig:           return the plotly figure (assign it, e.g. fig = draw_particles(...), to avoid
                          Jupyter displaying it twice)
    """
    rows = list(rows)
    show_other_particles = show_other_particles or show_all_activity
    visible = {"vertex": show_vertex, "other": show_other_particles, "activity": show_all_activity}
    filters_on = {"track": tracks_only, "throughgoing": throughgoing_only}
    build = {k: v or toggles for k, v in visible.items()}

    legend = []
    tags = []  # one per trace: {"track": bool|None, "throughgoing": bool|None}; None = not filtered
    fig = go.Figure()
    title_string = given_title or f"Matched particles (particle_df rows): {[int(r) for r in rows]}"
    setup_figure(fig, title_string, legend)
    tag_traces(fig, tags, None, None)  # detector boxes
    for item in legend:
        item["layer"] = "detector"

    layers = {}  # key -> list of trace indices

    def drawn_as(key, label, draw):
        """Run draw(); tag the traces and legend rows it added with layer key / label."""
        t0, l0 = len(fig.data), len(legend)
        draw()
        layers.setdefault(key, []).extend(range(t0, len(fig.data)))
        for item in legend[l0:]:
            item["layer"] = label

    chosen = particle_df.loc[rows]
    vertices = chosen[["SPINE_event", "ixn_id", "TS_entry", "TS_vtx_global_id"]].drop_duplicates()

    for c, r in enumerate(rows):
        drawn_as("chosen", "chosen particles",
                 lambda: plot_matched_particle(fig, particle_df.loc[r], c, False, legend, tags))
    if build["vertex"]:
        for v in vertices.itertuples():
            ev, ixn_id, gid = int(v.SPINE_event), int(v.ixn_id), int(v.TS_vtx_global_id)
            drawn_as("vertex", "vertex markers",
                     lambda: (add_vertex_markers(fig, ev, ixn_id, int(v.TS_entry), gid,
                                                 f"(spill {ev} ixn {ixn_id})", legend),
                              tag_traces(fig, tags, None, gid >= THROUGHGOING_FLAG)))
    others = particle_df.iloc[0:0]
    if build["other"]:
        others = particle_df[particle_df.set_index(["SPINE_event", "ixn_id"]).index.isin(
            list(zip(vertices.SPINE_event, vertices.ixn_id))) & ~particle_df.index.isin(rows)]
        for c, r in enumerate(others.index, start=len(rows)):
            drawn_as("other", "other particles at these vertices",
                     lambda: plot_matched_particle(fig, particle_df.loc[r], c, False, legend, tags))
        for v in vertices.itertuples():
            drawn_as("other", "other particles at these vertices",
                     lambda: plot_unmatched_at_vertex(fig, int(v.SPINE_event), int(v.ixn_id), int(v.TS_entry),
                                                      int(v.TS_vtx_global_id), legend, tags))
    if build["activity"]:
        drawn = particle_df.loc[rows + list(others.index)]
        skip_reco = {t for tracks in drawn.reco_tracks for t in tracks}
        if not build["other"]:
            # the other matched particles' reco tracks are not drawn by any layer, so keep them here
            skip_reco = {t for tracks in chosen.reco_tracks for t in tracks}
        drawn_as("activity", "all other activity in spill",
                 lambda: plot_other_activity(fig, sorted(set(chosen.SPINE_event)),
                                             skip_vertices=set(zip(vertices.SPINE_event, vertices.ixn_id)),
                                             skip_reco_tracks=skip_reco, legend=legend, tags=tags))
    assert len(tags) == len(fig.data) == len(legend)

    # Layer of each trace, and whether each filter hides it (filter on, and trace tagged False for it)
    layer_of = {i: key for key, idx in layers.items() for i in idx}
    filtered = {f: [i for i, t in enumerate(tags) if t[f] is False] for f, _ in TOGGLE_FILTERS}

    def shown_at_start(i):
        return (visible.get(layer_of.get(i), True)
                and not any(filters_on[f] and i in set(filtered[f]) for f in filters_on))

    for item, t, i in zip(legend, tags, range(len(legend))):
        item["track"] = {True: "yes", False: "no", None: "–"}[t["track"]]
        item["throughgoing"] = {True: "yes", False: "no", None: "–"}[t["throughgoing"]]
        item["visible at start"] = "yes" if shown_at_start(i) else "no"

    if not toggles:
        # nothing can be switched on later: drop what starts hidden
        keep = [i for i in range(len(fig.data)) if shown_at_start(i)]
        fig.data = tuple(fig.data[i] for i in keep)
        legend = [legend[i] for i in keep]
    else:
        # Layers use "visible". The filters use other properties so all buttons combine (a trace shows only
        # if its layer is on and no filter hides it): tracks -> trace opacity, throughgoing -> marker
        # opacity + line width.
        for key, _ in TOGGLE_LAYERS:
            for i in layers.get(key, []):
                fig.data[i].visible = visible[key]
        restore = {
            "track": {"opacity": [fig.data[i].opacity if fig.data[i].opacity is not None else 1
                                  for i in filtered["track"]]},
            "throughgoing": {"marker.opacity": [fig.data[i].marker.opacity if fig.data[i].marker.opacity is not None
                                                else 1 for i in filtered["throughgoing"]],
                             "line.width": [fig.data[i].line.width if fig.data[i].line.width is not None else 2
                                            for i in filtered["throughgoing"]]},
        }
        hide = {"track": {"opacity": 0}, "throughgoing": {"marker.opacity": 0, "line.width": 0}}
        for f, _ in TOGGLE_FILTERS:
            if filters_on[f]:
                for i in filtered[f]:
                    for k, v in hide[f].items():
                        fig.data[i].update({k.split(".")[0]: {k.split(".")[1]: v}} if "." in k else {k: v})

        menus = []

        def button_menu(n, row_y, label, idx, args_on, args_off, start_on):
            return dict(type="buttons", direction="right", showactive=True, active=-1,
                        x=0.02 + 0.2 * n, xanchor="left", y=row_y, yanchor="bottom", pad=dict(t=4, b=4),
                        buttons=[dict(label=f"{label} on/off", method="restyle",
                                      args=[args_off if start_on else args_on, idx],     # first click
                                      args2=[args_on if start_on else args_off, idx])])  # second click

        for n, (key, label) in enumerate(k for k in TOGGLE_LAYERS if layers.get(k[0])):
            menus.append(button_menu(n, 1.06, label, layers[key], {"visible": True}, {"visible": False}, visible[key]))
        for n, (f, label) in enumerate(k for k in TOGGLE_FILTERS if filtered[k[0]]):
            menus.append(button_menu(n, 1.0, label, filtered[f], hide[f], restore[f], filters_on[f]))
        fig.update_layout(updatemenus=menus, margin=dict(t=150))
    fig.update_layout(showlegend=show_legend)
    fig.show()
    if show_table:
        show_legend_table(legend)
    if save_html:
        save_html_page(fig, save_html, legend)
    if return_fig:
        return fig


In [31]:
# Choose particle_df rows to plot (default: first few particles with a TMS reco track)
spill_num = 0
spill_mask = (particles_reco.SPINE_event == spill_num)
print("Matched Particles in Spill: ", sum(spill_mask))
print("Number of throughgoing in spill: ", sum(particles_reco.throughgoing[spill_mask]))

particle_rows_to_plot = list(particles_reco.index[spill_mask]) if len(particles_reco) else list(particle_df.index[:1])
draw_particles(particle_rows_to_plot, show_vertex=False, show_legend=False,
               show_other_particles=True, given_title="Matched particles in spill "+str(spill_num),
              save_html="spill"+str(spill_num)+"_all_activity.html")

Matched Particles in Spill:  30
Number of throughgoing in spill:  20


,colour,marker,item,shown as,count,layer,visible at start
,blue,box outline,TMS detector,detector volume,,detector,yes
,black,box outline,ND-LAr detector,detector volume,,detector,yes
,#E69F00,points,"(spill 0 ixn 4 track 0, PDG 13)",ND-LAr points,853,chosen particles,yes
,#f0c975,points,"(spill 0 ixn 4 track 0, PDG 13)",TMS true hits (41),41,chosen particles,yes
,#E69F00,line + diamonds,"(spill 0 ixn 4 track 0, PDG 13)","TMS reco track (slice 9, #0, 36 hits)",36,chosen particles,yes
,#0072B2,points,"(spill 0 ixn 5 track 0, PDG 13)",ND-LAr points,1435,chosen particles,yes
,#70b1d9,points,"(spill 0 ixn 5 track 0, PDG 13)",TMS true hits (84),84,chosen particles,yes
,#0072B2,line + diamonds,"(spill 0 ixn 5 track 0, PDG 13)","TMS reco track (slice 12, #0, 60 hits)",60,chosen particles,yes
,#009E73,points,"(spill 0 ixn 9 track 3, PDG 211)",ND-LAr points,59,chosen particles,yes
,#7ecdb9,points,"(spill 0 ixn 9 track 3, PDG 211)",TMS true hits (276),276,chosen particles,yes


Saved spill0_all_activity.html (7.4 MB)


In [30]:
for tsm in range(0,7):
    test_spill_mask = (particles_reco.SPINE_event == tsm)
    print("Spill: ", tsm)
    print("Matched Particles: ", sum(test_spill_mask), ", Vertices inside: ", sum(test_spill_mask)-sum(particles_reco.throughgoing[test_spill_mask]))

Spill:  0
Matched Particles:  30 , Vertices inside:  10
Spill:  1
Matched Particles:  37 , Vertices inside:  13
Spill:  2
Matched Particles:  35 , Vertices inside:  5
Spill:  3
Matched Particles:  32 , Vertices inside:  5
Spill:  4
Matched Particles:  32 , Vertices inside:  8
Spill:  5
Matched Particles:  34 , Vertices inside:  7
Spill:  6
Matched Particles:  28 , Vertices inside:  4
